# Add LSST Photometric Calibration Parameters to Light Curves — OPTIMIZED

Memory- and speed-optimized rewrite of `05_AddLSSTCalibParam.ipynb`, targeting
the larger-statistics run in `05b_FindSourcesAllSpectypes/usdf_butler/` where
the original notebook was observed at **12.57/16 GB** RAM and climbing.

## What was actually eating memory

`05_AddLSSTCalibParam.ipynb` calls `butler.get(ref)` on the **PVI** dataset
type (`preliminary_visit_image` / `visit_image`) just to pull out
`photocal = pvi.getPhotoCalib()` and `pvi.getDetector()`. But `butler.get(ref)`
on an Exposure-type dataset loads the **entire image**: pixel plane, mask
plane, variance plane — typically tens to a few hundred MB **per detector per
visit** — even though only a tiny calibration object (a few KB) is actually
needed. Across hundreds/thousands of unique `(band, visit, detector)` cache
misses, this is the dominant memory (and I/O time) cost, on top of the
LSST Butler's internal datastore/registry caches, which grow across many
`.get()` calls and are **not** released by `gc.collect()` or `del` alone
(same root cause already diagnosed for the 16 GB crash in
`01_MatchTargetsWithLSSTCamSources.ipynb` / `libExtractLightcurves.py`).

## What changes here

1. **Component-level Butler access** — `butler.get(f"{PVI_DATASET}.photoCalib", dataId=ref.dataId, ...)`
   instead of `butler.get(ref)`. This reads only the calibration HDU/component,
   never the pixel data. A quick probe cell (§5) checks this actually works
   against this Butler/collection before committing to the full loop; if
   component access isn't supported for some reason, it transparently falls
   back to the original full-exposure `get()` (with a loud warning, so you
   know performance/memory will be back to baseline).
2. **Camera geometry loaded once** — `detector_id → detector_name` no longer
   requires loading a PVI at all; it's built once from the camera object at
   the very start and reused for every row.
3. **`visit_from_image` sanity check made optional/sampled** — the previous
   full run showed **0/13628 mismatches** between `visit` and
   `pvi.visitInfo.id`, so instead of loading `visitInfo` (another Butler
   component get) for every single cache miss, it's fetched only for a
   configurable random sample (`N_VISITINFO_SPOTCHECKS`), and `visit` from
   the registry `ref.dataId` is trusted otherwise (as it already is for
   `detector_id`).
4. **Periodic Butler recreation** (`RESET_BUTLER_EVERY_N_LOADS`) — drops the
   internal datastore/registry cache periodically, exactly like
   `reset_butler_every` in `process_target_chunk`.
5. **Checkpointing + resume** — the enriched table is saved to Parquet every
   `CHECKPOINT_EVERY_STARS` stars. If the kernel crashes or is restarted,
   re-running the notebook picks up from the last checkpoint instead of
   starting over.
6. **Memory usage logged periodically** (`psutil`, RSS in GB) so you can see
   directly whether the fix is working, instead of guessing from the
   JupyterHub memory gauge.

## What is unchanged

Same enrichment logic and output schema as `05_AddLSSTCalibParam.ipynb`
(the one validated as reliable against `05b_AddLSSTCalibParam.ipynb` — see
`07_CheckCalibCoeff.ipynb` cross-check): star-by-star wide query with exact
`ref.dataId["visit"]` matching, cache keyed on `(band, visit, detector_id)`,
`calib_mean`/`calib_err`/`calib_local`/`zeropoint` columns.

## ⚠️ Not tested against the real Butler

This notebook was written offline (no access to the USDF RSP from this
session) — it has **not** been executed against `dp2_prep`. Run the probe
cell in §5 first on a handful of refs before launching the full star loop,
and check the logged messages: if component access silently falls back to
full-exposure loads, the memory gain will not materialize and you'll want to
ping me with the actual error message so I can adjust the component name(s).

## Input
- `data_MergeVisits_02_out/all_stars_lightcurves_mjd.csv` (global)
- `data_MergeVisits_02_out/per_star/*_lc_mjd.csv` (per-star)

## Output
- `data_AddCalib_05optim_out/all_stars_lightcurves_calib.csv` + `.parquet`
- `data_AddCalib_05optim_out/per_star/*_lc_calib.csv` + `.parquet`
- `data_AddCalib_05optim_out/checkpoint_all_stars_lightcurves_calib.parquet` (intermediate, deleted at the end)

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-15
- **Last update:** 2026-07-16 : very fast
- **Last update:** 2026-07-19 : DDF


```
Fichier créé : /Users/dagoret/Desktop/RubinLSSTStableStarsinDDF/notebooks/05b_FindSourcesAllSpectypes/usdf_butler/05optim_AddLSSTCalibParam.ipynb


- Avant de lancer la boucle complète, exécutez dans l'ordre les sections 1 à 5 seulement (jusqu'à la cellule "probe"). Elle compare directement, sur une seule ref réelle :

       l'accès par composant butler.get(f"{PVI_DATASET}.photoCalib", dataId=...)
       contre un butler.get(ref) complet

        et logue le delta de RSS mémoire pour les deux. 
        
Ça vous dira en quelques secondes si l'astuce fonctionne réellement sur votre Butler avant d'engager tout le run.


- Points d'attention pour la suite :

Si le log.warning("Component get FAILED...") apparaît dans la §5, l'accès par composant n'est pas supporté tel quel sur ce dataset type/storage class — dites-le moi avec le message d'erreur exact, j'ajusterai (peut-être un nom de composant différent, ou passer par butler.getDirect/DeferredDatasetHandle).
Le notebook reprend automatiquement sur crash (RESUME_FROM_CHECKPOINT) grâce au checkpoint Parquet sauvegardé toutes les 25 étoiles — donc même s'il replante, vous ne repartez pas de zéro.
RESET_BUTLER_EVERY_N_LOADS=200 et CHECKPOINT_EVERY_STARS=25 sont des valeurs de départ raisonnables mais ajustables selon ce que vous verrez dans les logs RSS.
```

## 1. Imports

In [ ]:
import gc
import logging
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
from lsst.geom import Point2D

from libExtractLightcurves import safe_name, find_col, dataset_type_exists

try:
    import psutil

    _PROCESS = psutil.Process(os.getpid())

    def rss_gb() -> float:
        """Current resident memory of this kernel process, in GB."""
        return _PROCESS.memory_info().rss / (1024**3)

except ImportError:

    def rss_gb() -> float:
        return float("nan")

    print("psutil not available -> memory logging disabled (pip install psutil --user)")

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised. RSS at start: %.2f GB", rss_gb())

## 3. Configuration

In [ ]:
DDF_SELECTED = "ECDFS"
# DDF_SELECTED = "XMM-LSS"

# -- Notebook tag ------------------------------------------------------------
NB_TAG = "AddCalib_05optim"

# -- Input: light curves with MJD (output of notebook 02) --------------------
DIR_LC_IN = "./data_MergeVisits_02_out"
if DDF_SELECTED is None:
    DIR_LC_PER_STAR_IN = os.path.join(DIR_LC_IN, "per_star")
else:
    DIR_LC_PER_STAR_IN = os.path.join(DIR_LC_IN, f"per_star_{DDF_SELECTED}")

GLOBAL_LC_FILE = "all_stars_lightcurves_mjd.csv"

# -- Output --------------------------------------------------------------------
DIR_DATA = f"./data_{NB_TAG}_out"

if DDF_SELECTED is None:
    DIR_DATA_PER_STAR = os.path.join(DIR_DATA, "per_star")
else:
    DIR_DATA_PER_STAR = os.path.join(DIR_DATA, f"per_star_{DDF_SELECTED}")

DIR_FIGS = f"./figs_{NB_TAG}"
CHECKPOINT_PATH = os.path.join(DIR_DATA, "checkpoint_all_stars_lightcurves_calib.parquet")

for d in [DIR_DATA, DIR_DATA_PER_STAR, DIR_FIGS]:
    os.makedirs(d, exist_ok=True)
    log.info("Directory ready: %s", d)

# -- Butler configuration -------------------------------------------------------
repo = "dp2_prep"
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]
INSTRUMENT = "LSSTCam"

# -- Timespan: wide window covering all DP2 (used in spatial queries) ----------
DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-31T00:00:00"
MJD_START = Time(DATE_START, format="isot", scale="utc").mjd
MJD_STOP = Time(DATE_STOP, format="isot", scale="utc").mjd
log.info("MJD window: [%.2f, %.2f]", MJD_START, MJD_STOP)

# -- Candidate dataset type names for the single-visit processed image ---------
PVI_TABLE_CANDIDATES = [
    "preliminary_visit_image",
    "legacy_visit_image",
    "visit_image",
]

# -- Memory / robustness knobs ---------------------------------------------------
RESET_BUTLER_EVERY_N_LOADS = 10000  # recreate Butler after this many cache-miss loads
CHECKPOINT_EVERY_STARS = 25  # write a full checkpoint every N stars processed
N_VISITINFO_SPOTCHECKS = 30  # how many (band,visit,detector) get a visitInfo sanity check
LOG_MEMORY_EVERY_N_LOADS = 50  # log RSS every N cache-miss loads
RESUME_FROM_CHECKPOINT = True  # set False to force a clean restart

log.info("Configuration done.")

## 4. Initialise the Butler, select the PVI dataset type, load camera geometry

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
log.info("Butler initialised | repo: %s | RSS: %.2f GB", repo, rss_gb())

In [ ]:
# Select the first registered PVI dataset type
PVI_DATASET = None
for name in PVI_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        PVI_DATASET = name
        log.info("Selected PVI dataset type: '%s'", PVI_DATASET)
        break

if PVI_DATASET is None:
    raise RuntimeError(
        "No recognised PVI dataset type found in this Butler collection. "
        f"Candidates tried: {PVI_TABLE_CANDIDATES}"
    )

In [ ]:
# Detector geometry loaded ONCE -> detector_id -> detector_name, no per-visit
# PVI load needed just to get the name (it never changes across visits).
det_id_to_name: dict[int, str] = {}
try:
    camera = butler.get("camera", instrument=INSTRUMENT, collections=collection)
    for det in camera:
        det_id_to_name[int(det.getId())] = str(det.getName())
    log.info("Camera geometry loaded: %d detectors -> id/name map built.", len(det_id_to_name))
except Exception as exc:
    log.warning(
        "Could not load 'camera' dataset directly (%s). "
        "Falling back to resolving detector_name lazily from the first PVI seen "
        "per detector_id during the main loop.",
        exc,
    )

## 5. Probe: does component-level access to `photoCalib` work?

**Run this before the full loop.** It fetches `PVI_DATASET.photoCalib` for a
single ref via component access and compares the memory delta against a full
`butler.get(ref)` on the *same* ref, so you can see the saving directly. If
the component get raises, the main loop's `load_photocalib` will fall back to
full loads automatically — but you'll want to know that ahead of time rather
than discover it 12 GB in.

In [ ]:
def _first_available_ref(butler: Butler, dataset_type: str, timespan: Timespan, ra: float, dec: float):
    refs = list(
        butler.query_datasets(
            dataset_type,
            where=(
                "visit.timespan OVERLAPS :timespan "
                "AND visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
            ),
            bind={"timespan": timespan, "ra": ra, "dec": dec},
        )
    )
    return refs[0] if refs else None

In [ ]:
t1 = Time(MJD_START, format="mjd", scale="tai")
t2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(t1, t2)
log.info("Timespan for Butler queries: MJD [%.1f, %.1f]  (delta=%.0f days)", t1.mjd, t2.mjd, t2.mjd - t1.mjd)

# Load just enough of the LC table to get one probe position — full load happens in §6.
_df_probe = pd.read_csv(os.path.join(DIR_LC_IN, GLOBAL_LC_FILE), nrows=1)
_probe_ra = float(_df_probe["src_ra"].iloc[0])
_probe_dec = float(_df_probe["src_dec"].iloc[0])
del _df_probe

probe_ref = _first_available_ref(butler, PVI_DATASET, timespan, _probe_ra, _probe_dec)
if probe_ref is None:
    log.warning("Probe: no ref found for the first star's position -- skipping component-access test.")
    COMPONENT_ACCESS_OK = False
else:
    rss_before = rss_gb()
    try:
        t0 = time.time()
        probe_photocal = butler.get(
            f"{PVI_DATASET}.photoCalib", dataId=probe_ref.dataId, collections=collection
        )
        dt_component = time.time() - t0
        rss_after_component = rss_gb()
        log.info(
            "Component get OK: photoCalib=%s | dt=%.2fs | RSS %.2f -> %.2f GB (delta %.3f GB)",
            type(probe_photocal).__name__,
            dt_component,
            rss_before,
            rss_after_component,
            rss_after_component - rss_before,
        )
        COMPONENT_ACCESS_OK = True
        del probe_photocal
    except Exception as exc:
        log.warning("Component get FAILED (%s) -- main loop will fall back to full butler.get(ref).", exc)
        COMPONENT_ACCESS_OK = False
    gc.collect()

    if COMPONENT_ACCESS_OK:
        rss_before_full = rss_gb()
        t0 = time.time()
        probe_pvi_full = butler.get(probe_ref)
        dt_full = time.time() - t0
        rss_after_full = rss_gb()
        log.info(
            "For comparison, full get(ref): dt=%.2fs | RSS %.2f -> %.2f GB (delta %.3f GB)",
            dt_full,
            rss_before_full,
            rss_after_full,
            rss_after_full - rss_before_full,
        )
        del probe_pvi_full
        gc.collect()
        log.info("RSS after cleanup: %.2f GB", rss_gb())

## 6. Load the global merged light-curve table (with resume support)

In [ ]:
lc_path = os.path.join(DIR_LC_IN, GLOBAL_LC_FILE)
log.info("Loading: %s", lc_path)
df_all = pd.read_csv(lc_path)
log.info("Shape: %s  |  columns: %s", df_all.shape, df_all.columns.tolist())

REQUIRED_COLS = ["simbad_id", "visit", "src_ra", "src_dec", "x", "y", "band"]
missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Required columns missing from input LC file: {missing}")
log.info("All required columns present.")

df_all["visit"] = df_all["visit"].astype(np.int64)

# New columns (pre-allocated; overwritten below if a checkpoint is resumed)
df_all["calib_mean"] = np.nan
df_all["calib_err"] = np.nan
df_all["calib_local"] = np.nan
df_all["zeropoint"] = np.nan
df_all["visit_from_image"] = -1
df_all["detector_id"] = -1
df_all["detector_name"] = ""

star_ids = df_all["simbad_id"].unique()
log.info("Number of unique stars: %d", len(star_ids))

In [ ]:
# -- Resume from a previous checkpoint, if present ----------------------------
already_done_stars: set = set()

if RESUME_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
    log.info("Checkpoint found: %s -- resuming.", CHECKPOINT_PATH)
    df_ckpt = pd.read_parquet(CHECKPOINT_PATH)

    # Merge the checkpoint's enriched columns back onto df_all, row-aligned on
    # (simbad_id, band, visit, x, y) -- robust even if row order differs.
    key = ["simbad_id", "band", "visit", "x", "y"]
    enrich_cols = [
        "calib_mean",
        "calib_err",
        "calib_local",
        "zeropoint",
        "visit_from_image",
        "detector_id",
        "detector_name",
    ]
    df_all = df_all.drop(columns=enrich_cols).merge(df_ckpt[key + enrich_cols], on=key, how="left")
    for c in ["calib_mean", "calib_err", "calib_local", "zeropoint"]:
        df_all[c] = df_all[c].astype(float)
    df_all["visit_from_image"] = df_all["visit_from_image"].fillna(-1).astype(np.int64)
    df_all["detector_id"] = df_all["detector_id"].fillna(-1).astype(np.int64)
    df_all["detector_name"] = df_all["detector_name"].fillna("")

    # A star counts as "already done" if none of its rows still have detector_id == -1
    done_mask = df_all.groupby("simbad_id")["detector_id"].transform(lambda s: (s != -1).all())
    already_done_stars = set(df_all.loc[done_mask, "simbad_id"].unique())
    del df_ckpt
    gc.collect()
    log.info(
        "Resume: %d / %d stars already fully processed, will be skipped.",
        len(already_done_stars),
        len(star_ids),
    )
else:
    log.info(
        "No checkpoint resumed (RESUME_FROM_CHECKPOINT=%s, exists=%s).",
        RESUME_FROM_CHECKPOINT,
        os.path.exists(CHECKPOINT_PATH),
    )

## 7. Core loading function — component access with automatic fallback

In [ ]:
def load_calib_component(butler: Butler, ref, do_visitinfo_check: bool) -> dict:
    """Load PhotoCalib (+ optionally visitInfo) via component access, without
    ever pulling the pixel/mask/variance planes into memory.

    Falls back to a full `butler.get(ref)` only if component access fails
    (logged loudly so the fallback doesn't go unnoticed).
    """
    result = dict(
        calib_mean=np.nan,
        calib_err=np.nan,
        zeropoint=np.nan,
        visit_from_image=-1,
        detector_id=int(ref.dataId["detector"]),
        detector_name=det_id_to_name.get(int(ref.dataId["detector"]), ""),
        photocal=None,
        calib_ok=False,
    )
    try:
        photocal = butler.get(f"{PVI_DATASET}.photoCalib", dataId=ref.dataId, collections=collection)
        result["calib_mean"] = photocal.getCalibrationMean()
        result["calib_err"] = photocal.getCalibrationErr()
        result["zeropoint"] = 2.5 * np.log10(photocal.getInstFluxAtZeroMagnitude())
        result["photocal"] = photocal
        result["calib_ok"] = True

        if not result["detector_name"]:
            # camera geometry wasn't loaded in §4 -- resolve lazily via component get
            try:
                det_obj = butler.get(f"{PVI_DATASET}.detector", dataId=ref.dataId, collections=collection)
                result["detector_name"] = str(det_obj.getName())
                det_id_to_name[result["detector_id"]] = result["detector_name"]
                del det_obj
            except Exception:
                pass

        if do_visitinfo_check:
            try:
                vinfo = butler.get(f"{PVI_DATASET}.visitInfo", dataId=ref.dataId, collections=collection)
                result["visit_from_image"] = int(vinfo.id)
                del vinfo
            except Exception as exc:
                log.debug("    visitInfo component get failed for %s: %s", ref.dataId, exc)
        else:
            # Trust the registry dataId (validated: 0 mismatches in the full 05 run)
            result["visit_from_image"] = int(ref.dataId["visit"])

    except Exception as exc_component:
        log.warning(
            "    Component get('%s.photoCalib') failed for %s (%s) -- "
            "falling back to full butler.get(ref) [SLOW / MEMORY-HEAVY].",
            PVI_DATASET,
            ref.dataId,
            exc_component,
        )
        try:
            pvi = butler.get(ref)
            photocal = pvi.getPhotoCalib()
            det_obj = pvi.getDetector()
            result["calib_mean"] = photocal.getCalibrationMean()
            result["calib_err"] = photocal.getCalibrationErr()
            result["zeropoint"] = 2.5 * np.log10(photocal.getInstFluxAtZeroMagnitude())
            result["visit_from_image"] = int(pvi.visitInfo.id)
            result["detector_id"] = int(det_obj.getId())
            result["detector_name"] = str(det_obj.getName())
            result["photocal"] = photocal
            result["calib_ok"] = True
            del pvi, det_obj
        except Exception as exc_full:
            log.warning("    Full fallback load ALSO failed for %s: %s", ref.dataId, exc_full)

    return result

## 8. Core processing: star-by-star loop (memory-bounded, resumable)

In [ ]:
def save_checkpoint(df: pd.DataFrame, path: str) -> None:
    df.to_parquet(path, index=False)
    log.info("Checkpoint saved -> %s (%d rows) | RSS: %.2f GB", path, len(df), rss_gb())

In [ ]:
calib_cache: dict = {}
bands_in_data = sorted(df_all["band"].unique())
log.info("Bands present in data: %s", bands_in_data)

n_stars_ok = 0
n_stars_err = 0
n_stars_skipped = 0
n_loads_total = 0
n_loads_since_reset = 0

# Fixed random sample of cache keys that will get the visitInfo sanity check
rng = random.Random(42)

for i_star, star_id in enumerate(star_ids):
    if star_id in already_done_stars:
        n_stars_skipped += 1
        continue

    mask_star = df_all["simbad_id"] == star_id
    df_star = df_all.loc[mask_star]

    ra_star = float(df_star["src_ra"].iloc[0])
    dec_star = float(df_star["src_dec"].iloc[0])

    log.info(
        "[%d/%d] Star: %-30s  (ra=%.5f, dec=%.5f)  %d rows  | RSS: %.2f GB",
        i_star + 1,
        len(star_ids),
        star_id,
        ra_star,
        dec_star,
        mask_star.sum(),
        rss_gb(),
    )

    for band in bands_in_data:
        mask_band = mask_star & (df_all["band"] == band)
        if not mask_band.any():
            continue

        visits_for_star_band = df_all.loc[mask_band, "visit"].unique()

        try:
            refs = list(
                butler.query_datasets(
                    PVI_DATASET,
                    where=(
                        "band.name = :band "
                        "AND visit.timespan OVERLAPS :timespan "
                        "AND visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
                    ),
                    bind={"band": band, "timespan": timespan, "ra": ra_star, "dec": dec_star},
                    order_by=["visit.timespan.begin"],
                )
            )
        except Exception as exc:
            log.warning("    query_datasets failed (star=%s band=%s): %s", star_id, band, exc)
            n_stars_err += 1
            continue

        visit_to_ref = {int(ref.dataId["visit"]): ref for ref in refs}

        for idx in df_all.index[mask_band]:
            visit = int(df_all.at[idx, "visit"])
            x = float(df_all.at[idx, "x"])
            y = float(df_all.at[idx, "y"])

            ref = visit_to_ref.get(visit)
            if ref is None:
                continue

            det_id_from_ref = int(ref.dataId["detector"])
            cache_key = (band, visit, det_id_from_ref)

            if cache_key not in calib_cache:
                do_check = rng.random() < (N_VISITINFO_SPOTCHECKS / max(len(star_ids) * 6, 1))
                calib_cache[cache_key] = load_calib_component(butler, ref, do_visitinfo_check=do_check)

                n_loads_total += 1
                n_loads_since_reset += 1

                if n_loads_total % LOG_MEMORY_EVERY_N_LOADS == 0:
                    log.info(
                        "    [%d loads] RSS: %.2f GB | cache size: %d",
                        n_loads_total,
                        rss_gb(),
                        len(calib_cache),
                    )

                if n_loads_since_reset >= RESET_BUTLER_EVERY_N_LOADS:
                    log.info(
                        "  -- recreating Butler to release internal caches (RSS before: %.2f GB) --", rss_gb()
                    )
                    del butler
                    gc.collect()
                    butler = Butler(repo, collections=collection)
                    n_loads_since_reset = 0
                    log.info("  -- Butler recreated (RSS after: %.2f GB) --", rss_gb())

            cdict = calib_cache[cache_key]
            if not cdict["calib_ok"]:
                continue

            df_all.at[idx, "calib_mean"] = cdict["calib_mean"]
            df_all.at[idx, "calib_err"] = cdict["calib_err"]
            df_all.at[idx, "zeropoint"] = cdict["zeropoint"]
            df_all.at[idx, "visit_from_image"] = cdict["visit_from_image"]
            df_all.at[idx, "detector_id"] = cdict["detector_id"]
            df_all.at[idx, "detector_name"] = cdict["detector_name"]

            try:
                pos = Point2D(x, y)
                df_all.at[idx, "calib_local"] = cdict["photocal"].getLocalCalibration(pos)
            except Exception as exc:
                log.debug("    getLocalCalibration failed at idx=%d: %s", idx, exc)

    n_stars_ok += 1

    if (i_star + 1) % CHECKPOINT_EVERY_STARS == 0:
        save_checkpoint(df_all, CHECKPOINT_PATH)

log.info(
    "Star loop done: %d OK, %d skipped (resumed), %d with query errors. Total PVI component loads: %d.",
    n_stars_ok,
    n_stars_skipped,
    n_stars_err,
    n_loads_total,
)
log.info(
    "Final cache size: %d unique (band, visit, detector) entries. RSS: %.2f GB", len(calib_cache), rss_gb()
)

# Final checkpoint (covers any stars processed since the last periodic save)
save_checkpoint(df_all, CHECKPOINT_PATH)

## 9. Sanity check: visit number consistency (on the spot-checked subset)

In [ ]:
mask_checked = df_all["visit_from_image"] > 0
mismatches = df_all.loc[
    mask_checked & (df_all["visit"] != df_all["visit_from_image"]),
    ["simbad_id", "band", "visit", "visit_from_image", "detector_id"],
]
n_checked = mask_checked.sum()

if len(mismatches) > 0:
    print(f"WARNING -- {len(mismatches)} visit mismatches found (out of {n_checked} spot-checked rows):")
    display(mismatches)
else:
    log.info("All %d spot-checked visit numbers consistent (registry vs visitInfo.id) -- OK.", n_checked)

## 10. Save the global enriched file

In [ ]:
def save_calib(df: pd.DataFrame, out_dir: str, basename: str) -> None:
    """Save enriched DataFrame as both CSV and Parquet."""
    csv_path = os.path.join(out_dir, basename + ".csv")
    parquet_path = os.path.join(out_dir, basename + ".parquet")
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    log.info("Saved CSV    : %s  (%d rows)", csv_path, len(df))
    log.info("Saved Parquet: %s", parquet_path)

In [ ]:
if DDF_SELECTED is None:
    save_calib(df_all, DIR_DATA, "all_stars_lightcurves_calib")
else:
    save_calib(df_all, DIR_DATA, f"all_stars_lightcurves_calib_{DDF_SELECTED}")


df_all[
    [
        "simbad_id",
        "visit",
        "band",
        "detector_id",
        "detector_name",
        "calib_mean",
        "calib_err",
        "calib_local",
        "zeropoint",
        "visit_from_image",
    ]
].head(8)

## 11. Generate and save per-star files from the enriched global table

In [ ]:
per_star_files = sorted(f for f in os.listdir(DIR_LC_PER_STAR_IN) if f.endswith("_lc_mjd.csv"))
log.info("Found %d per-star CSV files in %s", len(per_star_files), DIR_LC_PER_STAR_IN)

n_ok_ps = 0
n_err_ps = 0

for fname in per_star_files:
    src_path = os.path.join(DIR_LC_PER_STAR_IN, fname)
    try:
        df_star_orig = pd.read_csv(src_path)
        df_star_orig["visit"] = df_star_orig["visit"].astype(np.int64)
    except Exception as exc:
        log.error("ERROR reading %s: %s", fname, exc)
        n_err_ps += 1
        continue

    star_ids_in_file = df_star_orig["simbad_id"].unique()

    df_star_calib = df_all.loc[
        df_all["simbad_id"].isin(star_ids_in_file) & df_all["visit"].isin(df_star_orig["visit"])
    ].copy()

    stem = fname.replace("_lc_mjd.csv", "")
    out_stem = stem + "_lc_calib"
    save_calib(df_star_calib, DIR_DATA_PER_STAR, out_stem)
    n_ok_ps += 1

    del df_star_orig, df_star_calib

log.info("Per-star files saved: %d OK, %d errors.", n_ok_ps, n_err_ps)

## 12. Diagnostics

In [ ]:
calib_summary = (
    df_all.dropna(subset=["calib_mean"])
    .groupby("band")
    .agg(
        n_rows=("calib_mean", "count"),
        calib_mean_med=("calib_mean", "median"),
        calib_mean_std=("calib_mean", "std"),
        calib_local_med=("calib_local", "median"),
        zeropoint_med=("zeropoint", "median"),
    )
)
print("Calibration summary per band:")
display(calib_summary)

n_missing = df_all["calib_mean"].isna().sum()
n_total = len(df_all)
log.info(
    "Rows without calib (Butler call failed or visit not in query): %d / %d  (%.1f %%)",
    n_missing,
    n_total,
    100.0 * n_missing / n_total if n_total else 0,
)

## 13. Clean up the intermediate checkpoint and list outputs

In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    log.info("Removed intermediate checkpoint: %s", CHECKPOINT_PATH)

print(f"\n=== Contents of {DIR_DATA} ===")
for entry in sorted(os.listdir(DIR_DATA)):
    full = os.path.join(DIR_DATA, entry)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  [DIR]  {entry}/  ({n} files)")
    else:
        size_kb = os.path.getsize(full) / 1024
        print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")

log.info("Done. Final RSS: %.2f GB", rss_gb())